# Phone Price Project — Combined Notebook by Contributor

This notebook brings the three contributor notebooks together in project order. Individual notebooks remain the source files; this copy is for reading and running the full sequence in one place.

## Contributor map

- **Viola:** initial inspection and exploratory split preview.
- **Amr:** S0 shared normalization, grouped splits, baseline metrics and experiment log.
- **Fadwa:** original S1 brand/model extraction rules, including work through Lenovo and LG.
- **Project integration additions:** unified `model_key`, `match_rule`, gold-set template and calibrated confidence workflow. These additions were made during repository cleanup and are identified separately below.

## How to use the baseline and metrics harness

Run the baseline after normalizing and splitting the data, before comparing more complex models. E0 predicts the training-fold global median; E1 predicts the training-fold median for each brand and falls back to the global median for unseen brands. Both use grouped out-of-fold predictions so duplicate title groups stay out of the validation fold. They give a reference score for deciding whether later models improve on a simple rule.

Use the metrics harness after a model has produced a prediction column. It calculates MAE and MedAE overall, by brand, and by price band; `add_to_log` records each experiment and `save_log` writes `experiment_log.csv`. It evaluates predictions; it does not train the model. Use grouped cross-validation or the development validation split during iteration. Keep the frozen test closed until the final evaluation.

```python
mae, medae = metrics(predictions, "pred_E0", price_band_cutoffs=price_band_cutoffs)
add_to_log(log, "E0", "global median", mae, medae)
save_log(log, output_dir / "experiment_log.csv")
```

## Run order

1. Put `dubizzle_phones_clean.csv` in `data/raw/` (or upload it when the notebook asks in Colab).
2. Run Viola’s section for exploration. It does not write split files.
3. Run Amr’s S0 section to create the split artifacts and baseline log.
4. Run Fadwa’s S1 rules and the integration section. The S1 section reads `train_set.csv`, leaves S0 files untouched, and creates a 200-row blank gold-label template.
5. Hand-label `gold_model_key` in the template, then rerun the S1 section to calculate precision, coverage, errors and rule confidence.


## Part 1 — Viola: initial data inspection

This section is Viola’s initial exploratory notebook. It previews the grouped split and does not write official project artifacts.


Imports

In [ ]:
import pandas as pd
import re

Load Data

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists() and (PROJECT_ROOT.parent / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dubizzle_phones_clean.csv"
if RAW_DATA_PATH.exists():
    df = pd.read_csv(RAW_DATA_PATH)
else:
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            f"Place dubizzle_phones_clean.csv at {RAW_DATA_PATH} or run in Colab and upload it."
        ) from exc
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV file was uploaded.")
    df = pd.read_csv(next(iter(uploaded)))


Dataset Shape

In [ ]:
print("Dataset shape:", df.shape)

Dataset Columns

In [ ]:
print("Columns names:\n", df.columns.to_list())

In [ ]:
print("Number of columns:", len(df.columns))

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().to_frame("Missing Values")

Data Types and Missing Values

In [ ]:
df.info()

In [ ]:
df.isna().sum().to_frame("Number of Missing Values")

Check Duplicate Titles

In [ ]:
df["title"].duplicated().sum()

Inspect Duplicate Titles

In [ ]:
df.loc[df["title"].duplicated(keep = False) , "title"].head(20).to_frame()

Inspect Title Examples

In [ ]:
df["title"].head(30)

Check Title Normalization Examples

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False), "title"]

Check Title Variations

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].nunique()

Inspect Unique Title Variations

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].drop_duplicates().to_list()

Title Normalization

In [ ]:
df["title_normalized"] = df["title"].str.lower().str.strip()

In [ ]:
df[["title", "title_normalized"]].head(20)

Compare Unique Titles

In [ ]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())

Normalize Arabic Letter Variants

In [ ]:
df["title_normalized"] = (df["title_normalized"]
    .str.replace("أ" , "ا", regex=False)
    .str.replace("إ" , "ا", regex=False)
    .str.replace("آ" , "ا", regex=False)
    .str.replace("ى" , "ي", regex=False)
)

Compare Normalized Titles

In [ ]:
df["title_normalized"].nunique()

Inspect Title Punctuation

In [ ]:
df["title_normalized"].head(20).to_list()

Punctuation Normalization

In [ ]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
    .str.split()
    .str.join(" ")
)

Compare Titles After Punctuation Normalization

In [ ]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Review Normalized Titles

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].head(10)

Improve Punctuation Normalization

In [ ]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

In [ ]:
df.loc[df["title_normalized"] != df["title_punct_normalized"],["title_normalized", "title_punct_normalized"]].head(10)

In [ ]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Normalize Extra Spaces

In [ ]:
df["title_punct_normalized"] = (
    df["title_punct_normalized"]
    .str.split()
    .str.join(" ")
)

In [ ]:
df["title_punct_normalized"].head(20).to_list()

Inspect Remaining Punctuation

In [ ]:
df.loc[df["title_punct_normalized"].str.contains(r"[|()]", regex=True, na=False), ["title", "title_punct_normalized"]].head(10)

Normalize Arabic and English Digits

In [ ]:
digit_map = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

df["title_normalized"] = (df["title_normalized"].str.translate(digit_map))

Remove Arabic Diacritics

In [ ]:
df["title_normalized"] = (df["title_normalized"].str.replace(r"[\u064B-\u065F\u0670]", "" , regex=True))

In [ ]:
df.loc[df["title"].str.contains("[ًٌٍَُِّْ]", na=False),["title", "title_normalized"]].head(10)

Compare Unique Titles Before and After Punctuation Normalization

In [ ]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

Rebuild Punctuation-Normalized Titles

In [ ]:
from preprocessing.normalization import normalize_title_for_group

df["title_normalized"] = df["title"].map(normalize_title_for_group)
df["title_punct_normalized"] = df["title_normalized"]


In [ ]:
df[["title", "brand", "storage_gb", "ram_gb"]].head(20)

In [ ]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

Check for Extra Spaces in Normalized Titles

In [ ]:
df["title_punct_normalized"].str.contains(r"\s{2,}", regex=True, na=False).sum()

 Verify Final Title Normalization

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

Count Unique Titles After Final Normalization

In [ ]:
df["title_punct_normalized"].nunique()

Create Final Normalized Title Groups

In [ ]:
df["title_group"] = df["title_punct_normalized"]


In [ ]:
print("Missing groups:", df["title_group"].isna().sum())
print("Total groups:", df["title_group"].nunique())
print("Total rows:", len(df))

Validate Title Normalization

In [ ]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())
print("Final title groups:", df["title_group"].nunique())
print("Missing groups:", df["title_group"].isna().sum())
print("Total rows:", len(df))

# Grouped Train-Test Split

In [ ]:
from preprocessing.splits import make_grouped_splits

preview_splits = make_grouped_splits(df)
dev_df = pd.concat(
    [preview_splits["train"], preview_splits["validation"], preview_splits["dev_test"]],
    axis=0,
).sort_index()
test_df = preview_splits["frozen_test"]


## Preview grouped split

In [ ]:
dev_groups = set(dev_df["title_group"])
test_groups = set(test_df["title_group"])
print("Development rows:", len(dev_df))
print("Preview test rows:", len(test_df))
print("Overlapping groups:", len(dev_groups & test_groups))


### Preview only — no project files are written

In [ ]:
print("This notebook is exploratory; it does not write project split files.")
print("Run the S0 notebook to create and lock the official split artifacts.")


In [ ]:
print("Development groups:", dev_df["title_group"].nunique())
print("Preview test groups:", test_df["title_group"].nunique())
print("Total groups:", df["title_group"].nunique())


In [ ]:
print("Preview development rows:", len(dev_df))
print("Preview test rows:", len(test_df))
print("Official split files were not changed.")


##### GroupKFold Cross-Validation

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(dev_df, groups=dev_df["title_group"]), start=1
):
    print(f"Fold {fold}: Train={len(train_idx)}, Validation={len(val_idx)}")


In [ ]:
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(dev_df, groups=dev_df["title_group"]), start=1
):
    train_groups = set(dev_df.iloc[train_idx]["title_group"])
    val_groups = set(dev_df.iloc[val_idx]["title_group"])
    print(f"Fold {fold}: Overlapping groups = {len(train_groups & val_groups)}")


## Part 2 — Amr: S0 split, baseline and metrics harness

This section creates the official development/frozen-test partitions and the first comparable baseline results.


Imports

In [ ]:
import pandas as pd
import re
import numpy as np 

Load Data

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists() and (PROJECT_ROOT.parent / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dubizzle_phones_clean.csv"
if RAW_DATA_PATH.exists():
    SOURCE_PATH = RAW_DATA_PATH
    df = pd.read_csv(SOURCE_PATH)
else:
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            f"Place dubizzle_phones_clean.csv at {RAW_DATA_PATH} or run in Colab and upload it."
        ) from exc
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV file was uploaded.")
    SOURCE_PATH = Path(next(iter(uploaded)))
    df = pd.read_csv(SOURCE_PATH)


Dataset Shape

In [ ]:
print("Dataset shape:", df.shape)

Dataset Columns

In [ ]:
print("Columns names:\n", df.columns.to_list())

In [ ]:
print("Number of columns:", len(df.columns))

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().to_frame("Missing Values")

Data Types and Missing Values

In [ ]:
df.info()

In [ ]:
df.isna().sum().to_frame("Number of Missing Values")

Check Duplicate Titles

In [ ]:
df["title"].duplicated().sum()

Inspect Duplicate Titles

In [ ]:
df.loc[df["title"].duplicated(keep = False) , "title"].head(20).to_frame()

Inspect Title Examples

In [ ]:
df["title"].head(30)

Check Title Normalization Examples

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False), "title"]

Check Title Variations

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].nunique()

Inspect Unique Title Variations

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].drop_duplicates().to_list()

Title Normalization

In [ ]:
df["title_normalized"] = df["title"].str.lower().str.strip()

In [ ]:
df[["title", "title_normalized"]].head(20)

Compare Unique Titles

In [ ]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())

Normalize Arabic Letter Variants

In [ ]:
df["title_normalized"] = (df["title_normalized"]
    .str.replace("أ" , "ا", regex=False)
    .str.replace("إ" , "ا", regex=False)
    .str.replace("آ" , "ا", regex=False)
    .str.replace("ى" , "ي", regex=False)
)

Compare Normalized Titles

In [ ]:
df["title_normalized"].nunique()

Inspect Title Punctuation

In [ ]:
df["title_normalized"].head(20).to_list()

Punctuation Normalization

In [ ]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
    .str.split()
    .str.join(" ")
)

Compare Titles After Punctuation Normalization

In [ ]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Review Normalized Titles

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].head(10)

Improve Punctuation Normalization

In [ ]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

In [ ]:
df.loc[df["title_normalized"] != df["title_punct_normalized"],["title_normalized", "title_punct_normalized"]].head(10)

In [ ]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Normalize Extra Spaces

In [ ]:
df["title_punct_normalized"] = (
    df["title_punct_normalized"]
    .str.split()
    .str.join(" ")
)

In [ ]:
df["title_punct_normalized"].head(20).to_list()

Inspect Remaining Punctuation

In [ ]:
df.loc[df["title_punct_normalized"].str.contains(r"[|()]", regex=True, na=False), ["title", "title_punct_normalized"]].head(10)

Normalize Arabic and English Digits

In [ ]:
digit_map = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

df["title_normalized"] = (df["title_normalized"].str.translate(digit_map))

Remove Arabic Diacritics

In [ ]:
df["title_normalized"] = (df["title_normalized"].str.replace(r"[\u064B-\u065F\u0670]", "" , regex=True))

In [ ]:
df.loc[df["title"].str.contains("[ًٌٍَُِّْ]", na=False),["title", "title_normalized"]].head(10)

Compare Unique Titles Before and After Punctuation Normalization

In [ ]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

Rebuild Punctuation-Normalized Titles

In [ ]:
from preprocessing.normalization import normalize_title_for_group

df["title_normalized"] = df["title"].map(normalize_title_for_group)
df["title_punct_normalized"] = df["title_normalized"]


In [ ]:
df[["title", "brand", "storage_gb", "ram_gb"]].head(20)

In [ ]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

Check for Extra Spaces in Normalized Titles

In [ ]:
df["title_punct_normalized"].str.contains(r"\s{2,}", regex=True, na=False).sum()

 Verify Final Title Normalization

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

Count Unique Titles After Final Normalization

In [ ]:
df["title_punct_normalized"].nunique()

Create Final Normalized Title Groups

In [ ]:
df["title_group"] = df["title_punct_normalized"]


In [ ]:
print("Missing groups:", df["title_group"].isna().sum())
print("Total groups:", df["title_group"].nunique())
print("Total rows:", len(df))

Validate Title Normalization

In [ ]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())
print("Final title groups:", df["title_group"].nunique())
print("Missing groups:", df["title_group"].isna().sum())
print("Total rows:", len(df))

# Grouped Train-Test Split

In [ ]:
from preprocessing.splits import make_grouped_splits, save_frozen_test_set

split_frames = make_grouped_splits(df, group_column="title_group", random_state=42)
train_df = split_frames["train"]
train_part = train_df  # baseline fitting is limited to this split
val_df = split_frames["validation"]
dev_test_df = split_frames["dev_test"]
test_df = split_frames["frozen_test"]


Validate Grouped Split

In [ ]:
train_groups = set(train_df["title_group"])
test_groups = set(test_df["title_group"])
print("Train rows:", len(train_df))
print("Final frozen-test rows:", len(test_df))
print("Overlapping groups:", len(train_groups & test_groups))


Save Frozen Test Set

In [ ]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
frozen_test_path = save_frozen_test_set(
    test_df, output_dir / "frozen_test_set.csv", id_column="ad_id"
)
print("Frozen test verified or created:", frozen_test_path)


In [ ]:
print("Train groups:", train_df["title_group"].nunique())
print("Test groups:", test_df["title_group"].nunique())
print("Total groups:", df["title_group"].nunique())

In [ ]:
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Development-test rows:", len(dev_test_df))
print("Frozen-test rows:", len(test_df))


##### GroupKFold Cross-Validation

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(train_df, groups=train_df["title_group"]), start=1
):
    print(f"Fold {fold}: Train={len(train_idx)}, Validation={len(val_idx)}")


In [ ]:
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(train_df, groups=train_df["title_group"]), start=1
):
    fold_train_groups = set(train_df.iloc[train_idx]["title_group"])
    fold_val_groups = set(train_df.iloc[val_idx]["title_group"])
    print(f"Fold {fold}: Overlapping groups = {len(fold_train_groups & fold_val_groups)}")


frozen to Data

In [ ]:
split_dir = PROJECT_ROOT / "data" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
print("Validation and development-test partitions came from the explicit seeded split.")


In [ ]:
print("Validation groups:", val_df["title_group"].nunique())
print("Development-test groups:", dev_test_df["title_group"].nunique())


In [ ]:
val_df.to_csv(split_dir / "validation.csv", index=False)
dev_test_df.to_csv(split_dir / "dev_test.csv", index=False)


In [ ]:
split_frames = {
    "train": train_df,
    "validation": val_df,
    "dev_test": dev_test_df,
    "frozen_test": test_df,
}


In [ ]:
train_df.to_csv(output_dir / "train_set.csv", index=False)
for left_name, left in split_frames.items():
    for right_name, right in split_frames.items():
        if left_name < right_name:
            overlap = set(left["title_group"]) & set(right["title_group"])
            print(f"{left_name}/{right_name} group overlap:", len(overlap))
print("Saved training rows:", len(train_df))

import hashlib
import json
source_hash = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
manifest = {
    "source_file": SOURCE_PATH.name,
    "source_sha256": source_hash,
    "normalizer": "preprocessing.normalization.normalize_title_for_group",
    "group_column": "title_group",
    "random_state": 42,
    "input_rows": int(len(df)),
    "unique_groups": int(df["title_group"].nunique()),
    "partitions": {
        name: {
            "rows": int(len(part)),
            "groups": int(part["title_group"].nunique()),
            "row_share": round(len(part) / len(df), 6),
        }
        for name, part in split_frames.items()
    },
}
(output_dir / "split_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("Split manifest saved:", output_dir / "split_manifest.json")


baseline

What is baseline ?

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

### Baseline implementation — Amr

E0 and E1 are trained inside each GroupKFold training fold and scored on its held-out groups. Use these scores as the benchmark for later experiments.


In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

# TODO: maybe clean this up later? works for now though.
def grouped_median_baseline(data, n_splits=5):
    # Just copying the dataframe so we don't mutate the original by accident (had bugs before because of this lol)
    predictions = data.copy()
    
    # Setting up arrays to hold our OOF predictions
    global_preds = np.full(len(data), np.nan)
    brand_preds = np.full(len(data), np.nan)
    
    gkf = GroupKFold(n_splits=n_splits)
    
    # Let's loop through the splits
    for train_idx, val_idx in gkf.split(data, groups=data["title_group"]):
        train_df = data.iloc[train_idx]
        val_df = data.iloc[val_idx]

        # just a sanity check to make sure group kfold is actually working as intended
        overlap = set(train_df["title_group"]) & set(val_df["title_group"])
        assert not overlap, f"Group leakage detected: {len(overlap)} overlapping groups"
            
        # calc baseline stats on train fold only!! very important to avoid leakage
        global_med = train_df["price"].median()
        
        # grouping by brand to find median price per brand
        brand_meds = train_df.groupby("brand")["price"].median()

        # Fill in global predictions
        global_preds[val_idx] = global_med
        
        # map the brand medians, and if a brand wasn't in train, fallback to global median
        mapped_brands = val_df["brand"].map(brand_meds)
        filled_brands = mapped_brands.fillna(global_med)
        
        brand_preds[val_idx] = filled_brands.to_numpy()

    # save back into df
    predictions["pred_E0"] = global_preds
    predictions["pred_E1"] = brand_preds
    
    # calculate final MAE 
    # Note: remember we are evaluating on the whole thing (OOF predictions combined)
    mae_global = mean_absolute_error(data["price"], predictions["pred_E0"])
    mae_brand = mean_absolute_error(data["price"], predictions["pred_E1"])
    
    scores = {
        "global_median_mae": mae_global,
        "brand_median_mae": mae_brand,
    }

    return predictions, scores

In [ ]:
baseline_predictions, baseline_scores = grouped_median_baseline(train_part)

In [ ]:
print("Global median MAE:", baseline_scores["global_median_mae"])
print("Brand median MAE:", baseline_scores["brand_median_mae"])

In [ ]:
import sys

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from evaluation.simple_harness import metrics, add_to_log, save_log
price_band_cutoffs = tuple(train_part["price"].quantile([1 / 3, 2 / 3]).to_list())
print("Price-band cutoffs from training data:", price_band_cutoffs)


In [ ]:
predictions, scores = baseline_predictions, baseline_scores


In [ ]:
log = []

mae, medae = metrics(predictions, "pred_E0", price_band_cutoffs=price_band_cutoffs)
add_to_log(log, "E0", "global median", mae, medae, notes=f"price bands: {price_band_cutoffs}")

mae, medae = metrics(predictions, "pred_E1", price_band_cutoffs=price_band_cutoffs)
add_to_log(log, "E1", "brand median", mae, medae, notes=f"price bands: {price_band_cutoffs}")

log_path = save_log(log, output_dir / "experiment_log.csv")
print("Experiment log saved:", log_path)


In [ ]:
print(df["price"].describe())
print(df["price"].quantile([0.33, 0.66]))

Imports

## Part 3 — Fadwa: S1 model-extraction rules

The brand-specific extraction rules in this section come from Fadwa’s notebook. The unified output, gold-label workflow and evaluation cells are later project-integration additions.


# S1 — Model-key extraction

Run S0 first. This notebook reads the S0 training partition, keeps its frozen test and split files untouched, and writes all S1 artifacts to `data/processed/stage1/`.


In [ ]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists() and (PROJECT_ROOT.parent / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from preprocessing.normalization import normalize_title_for_group

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_set.csv"
if not TRAIN_PATH.exists():
    raise FileNotFoundError("Run notebooks/01_Amr_S0_split_and_baseline.ipynb first to create train_set.csv.")
train_df = pd.read_csv(TRAIN_PATH)
required = {"ad_id", "title", "brand", "title_group"}
missing = required.difference(train_df.columns)
if missing:
    raise ValueError(f"S0 training file is missing required columns: {sorted(missing)}")
if train_df["ad_id"].duplicated().any():
    raise ValueError("ad_id must be unique in the S0 training partition")

stage1_output_dir = PROJECT_ROOT / "data" / "processed" / "stage1"
stage1_output_dir.mkdir(parents=True, exist_ok=True)
print("Loaded S0 training rows:", len(train_df))


In [ ]:
stage1_df = train_df.copy()

print("Stage 1 dataset shape:", stage1_df.shape)
print("Available columns:", stage1_df.columns.tolist())

Define Final iPhone Model Extraction Function


In [ ]:
def extract_iphone_model(title):
    text = re.sub(r"\s+", " ", str(title).lower()).strip()
    iphone_marker = re.search(r"(?:iphone|i\s*phone|ايفون|آيفون)", text)
    non_phone_product = re.search(
        r"\b(?:watch|airpods?|ipad|macbook|imac)\b|ساعة|ساعه|ايربودز|ايباد|ماكبوك",
        text,
    )
    if non_phone_product and not iphone_marker:
        return "unknown"
    special_cases = [
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*6\s*(?:s|اس|أس)\s*(?:plus|بلس)\b", "iphone 6s plus"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*x\s*(?:s|اس|أس)\s*(?:max|ماكس)\b", "iphone xs max"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*x\s*(?:max|ماكس)\b", "iphone x max"),
        (r"ip+h+o+n+e+\s*17\s+pro+m+a+x+", "iphone 17 pro max"),
        (r"ايفون+ن?\s*17\s*برماكس", "iphone 17 pro max"),
        (r"iphoneee?\s*17\s*pro+\s*ma+x+", "iphone 17 pro max"),
        (r"iphone+\s*17\s*pro+\s*ma+x+", "iphone 17 pro max"),
        (r"ايفون\s*17\s*برو+ما+كس", "iphone 17 pro max"),
        (r"ايفون\s*17\s*برماكس", "iphone 17 pro max"),
        (r"iphone\s*pro\s*max\s*17", "iphone 17 pro max"),
        (r"ايفون\s*pro\s*max\s*17", "iphone 17 pro max"),
        (r"iphone\s*3g\b", "iphone 3g"),
        (r"ايفون\s*3g\b", "iphone 3g"),
        (r"ايفون\s*6اس\b", "iphone 6s"),
        (r"ايفون\s*8بلص\b", "iphone 8 plus"),
        (r"ايفون\s*i\s*phone\s*plus\s*7", "iphone 7 plus"),
    ]

    for pattern, model in special_cases:
        if re.search(pattern, text):
            return model
    special_models = [
        (r"\bxs\s*max\b|\bاكس\s*اس\s*ماكس\b", "iphone xs max"),
        (r"\bxs\b|\bاكس\s*اس\b", "iphone xs"),
        (r"\bxr\b|\bاكس\s*ار\b", "iphone xr"),
        (r"\bx\b|\bاكس\b", "iphone x"),
        (r"\b(?:iphone|i\s*phone)\s*se\b", "iphone se"),
        (r"\bايفون\s*اس\s*اي\b", "iphone se"),
        (r"\bايفون\s*اكس\s*اس\s*ماكس\b", "iphone xs max"),
        (r"\bايفون\s*اكس\s*ار\b", "iphone xr"),
        (r"\bايفون\s*اكس\s*اس\b", "iphone xs"),
        (r"\bايفون\s*اكس\b", "iphone x"),
        (r"\b(?:iphone|i\s*phone)\s*xs\s*max\b", "iphone xs max"),
        (r"\b(?:iphone|i\s*phone)\s*xr\b", "iphone xr"),
        (r"\b(?:iphone|i\s*phone)\s*xs\b", "iphone xs"),
        (r"\b(?:iphone|i\s*phone)\s*x\b", "iphone x"),
    ]

    for pattern, model in special_models:
        if re.search(pattern, text):
            return model

    model_patterns = [
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*pro\s*max\b", "pro max"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*برو\s*ماكس\b", "pro max"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*plus\b", "plus"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*بلس\b", "plus"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*pro\b", "pro"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*برو\b", "pro"),
    ]

    for pattern, version in model_patterns:
        match = re.search(pattern, text)
        if match:
            return f"iphone {match.group(1)} {version}"

    numeric_pattern = re.compile(
        r"(?:\biphone\s*|\bi\s*phone\s*|ايفون\s*)"
        r"(\d{1,2})(s)?\s*"
        r"(pro\s*max|promax|pro|max|plus|mini|air|e)?\b"
    )

    match = numeric_pattern.search(text)

    if match:
        number = match.group(1)
        suffix = match.group(2) or ""
        version = re.sub(r"\s+", " ", (match.group(3) or "").strip())

        if version == "promax":
            version = "pro max"

        model = f"iphone {number}{suffix}"
        return f"{model} {version}" if version else model

    match = re.search(
        r"ايفون\s+(pro|max|plus|برو|ماكس|بلس)\s*(\d{1,2})",
        text
    )

    if match:
        version = {
            "برو": "pro",
            "ماكس": "max",
            "بلس": "plus"
        }.get(match.group(1), match.group(1))

        return f"iphone {match.group(2)} {version}"

    pattern = re.compile(
        r"\b(?:iphone\s*)?(\d{1,2})\s*"
        r"(pro\s*max|pro|max|plus|mini|air|e)?\b"
        r"|ايفون\s*(\d{1,2})\s*"
        r"(برو\s*ماكس|برو|ماكس|بلس|ميني|اير|e)?"
    )

    match = pattern.search(text)

    if not match:
        return "unknown"

    number = match.group(1) or match.group(3)
    version = (match.group(2) or match.group(4) or "").strip()

    version = {
        "برو ماكس": "pro max",
        "برو": "pro",
        "ماكس": "max",
        "بلس": "plus",
        "ميني": "mini",
        "اير": "air"
    }.get(version, version)

    return f"iphone {number} {version}".strip()

Extract iPhone Models and Evaluate Coverage


In [ ]:
apple_titles = stage1_df[stage1_df["brand"] == "apple"].copy()

apple_titles["iphone_model"] = apple_titles[
    "title_punct_normalized"
].apply(extract_iphone_model)

print("Total Apple ads:", len(apple_titles))
print("Extracted:", (apple_titles["iphone_model"] != "unknown").sum())
print("Unknown:", (apple_titles["iphone_model"] == "unknown").sum())
print(
    "Coverage:",
    round((apple_titles["iphone_model"] != "unknown").mean() * 100, 2),
    "%"
)

display(
    apple_titles[
        ["title", "title_punct_normalized", "iphone_model"]
    ].head(20)
)

Save Final iPhone Predictions

In [ ]:
iphone_model_compact = apple_titles["iphone_model"].copy()
print("Saved predictions:", len(iphone_model_compact))
print("Extracted:", (iphone_model_compact != "unknown").sum())
print("Unknown:", (iphone_model_compact == "unknown").sum())

Add iPhone Models to Stage 1 Dataset

In [ ]:
apple_mask = stage1_df["brand"] == "apple"
stage1_df["iphone_model"] = "not_apple"
stage1_df.loc[apple_mask, "iphone_model"] = (
    stage1_df.loc[apple_mask, "title_punct_normalized"].apply(extract_iphone_model)
)
stage1_df["iphone_match_rule"] = ""
stage1_df.loc[apple_mask & stage1_df["iphone_model"].ne("unknown"), "iphone_match_rule"] = "apple:ordered_regex"
print("iPhone column added successfully.")
print("Apple ads:", int(apple_mask.sum()))
print("Unknown iPhone models:", int((stage1_df.loc[apple_mask, "iphone_model"] == "unknown").sum()))
display(stage1_df.loc[apple_mask, ["title", "iphone_model"]].head(10))


Explore Samsung Listings

In [ ]:
samsung_titles = stage1_df[
    stage1_df["brand"] == "samsung"
].copy()

print("Total Samsung ads:", len(samsung_titles))

display(
    samsung_titles[
        ["title", "title_punct_normalized"]
    ].sample(
        n=min(20, len(samsung_titles)),
        random_state=10
    )
)

Define Final Samsung Model Extraction Function

In [ ]:
def extract_samsung_model(title):
    text = str(title).lower()
    text = re.sub(r"\s+", " ", text).strip()

    for ar, en in zip("٠١٢٣٤٥٦٧٨٩", "0123456789"):
        text = text.replace(ar, en)

    text = text.replace("⁰", "0").replace("⁵", "5").replace("⁷", "7")

    compact = re.sub(r"\s+", "", text)
    if "ztrifold" in compact or "ztri fold" in text or "tri fold" in text:
        return "samsung z trifold"

    if "سامسونجa36فجي" in compact or "سامسونجa365g" in compact:
        return "samsung a36"

    if "چى" in text and "7" in text and "برو" in text:
        return "samsung j7 pro"

    if "جراند" in text and "برايم" in text:
        if "برو" in text:
            return "samsung grand prime pro"
        return "samsung grand prime"

    if re.search(r"جيه\s*[٧7]\s*برايم", text):
        return "samsung j7 prime"
    replacements = {
        "سامسونج": " samsung ",
        "جالكسى": " galaxy ",
        "جالكسي": " galaxy ",
        "جالاكسي": " galaxy ",
        "جلاكسي": " galaxy ",
        "ايه": " a ",
        "إيه": " a ",
        "اي": " a ",
        "إي": " a ",
        "اس": " s ",
        "إس": " s ",
        "ام": " m ",
        "إم": " m ",
        "اف": " f ",
        "إف": " f ",
        "جيه": " j ",
        "جى": " g ",
        "چى": " g ",
        "نوت": " note ",
        "الترا": " ultra ",
        "بلس": " plus ",
        "برو": " pro ",
        "برايم": " prime ",
        "فولد": " fold ",
        "فليب": " flip ",
        "زد": " z ",
        "جراند": " grand ",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\b(?:samsung|galaxy)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    special_patterns = [
        (r"\bz\s*fold\s*(\d+)\b", "z fold"),
        (r"\bz\s*flip\s*(\d+)\b", "z flip"),
        (r"\b(?:samsung\s+)?fold\s*(\d+)\b", "fold"),
        (r"\bx\s*cover\s*(\d+)\s*(pro)?\b", "xcover"),
    ]

    for pattern, prefix in special_patterns:
        match = re.search(pattern, text)
        if match:
            model = prefix + match.group(1)
            if len(match.groups()) > 1 and match.group(2):
                model += " " + match.group(2)
            return "samsung " + model

    if re.search(r"\b26\s*ultra\b", text):
        return "samsung s26 ultra"

    if re.search(r"\ba\s*03\s*core\b", text):
        return "samsung a03 core"

    if re.search(r"\ba\s*04e\b", text):
        return "samsung a04e"

    if re.search(r"\ba\s*36\s*5g\b", text):
        return "samsung a36"

    if re.search(r"\ba\s*51\b", text):
        return "samsung a51"

    if re.search(r"\bflip\s*5\b", text):
        return "samsung z flip5"

    if re.search(r"\bgrand\s*prime\s*pro\b", text):
        return "samsung grand prime pro"

    if re.search(r"\bgrand\s*prime\b", text):
        return "samsung grand prime"

    if re.search(r"\bmega\b", text):
        return "samsung mega"

    patterns = [
        (r"\bf\s*(\d{1,3})\b", "f"),
        (r"\bnote\s*(\d+)\s*(ultra|plus)?\b", "note"),
        (r"\bnot\s*(\d+)\s*(ultra|plus)?\b", "note"),
        (r"\bs\s*(\d+)\s*(ultra|plus|fe)?\b", "s"),
        (r"\ba\s*(\d{1,3})\s*(ultra|plus|s)?\b", "a"),
        (r"\bm\s*(\d{1,3})\s*(ultra|plus|s)?\b", "m"),
        (r"\bj\s*(\d{1,2})\s*(prime|plus|pro)?\b", "j"),
    ]

    for pattern, series in patterns:
        match = re.search(pattern, text)
        if match:
            model = series + match.group(1)
            suffix = match.group(2) if len(match.groups()) > 1 else None

            if suffix:
                if series in ("a", "m") and suffix == "s":
                    model += "s"
                else:
                    model += " " + suffix

            return "samsung " + model

    match = re.search(r"\b(\d{1,2})\s*fe\b", text)
    if match:
        return "samsung s" + match.group(1) + " fe"

    match = re.search(r"\b(\d{1,3})\s*a\b", text)
    if match:
        return "samsung a" + match.group(1)

    return "unknown"

Test Special Samsung Model Cases

In [ ]:
tests = [
    "Galaxy Z TriFold",
    "سامسونجa36فجي تليفون حالته جميله",
    "Samsung 26 ultra",
    "سامسونج جلاكسي چى7برو",
    "Samsung Galaxy Z Fold 5",
    "Samsung Galaxy Z Flip 6",
    "Samsung Galaxy A03 Core",
    "Samsung Galaxy A04e",
]

for title in tests:
    print(f"{title} -> {extract_samsung_model(title)}")

 Extract Samsung Models and Evaluate Coverage

In [ ]:
samsung_titles = stage1_df.loc[
    stage1_df["brand"] == "samsung"
].copy()

samsung_titles["samsung_model"] = samsung_titles["title"].apply(
    extract_samsung_model
)

total = len(samsung_titles)
unknown = (samsung_titles["samsung_model"] == "unknown").sum()
known = total - unknown

print("Total Samsung ads:", total)
print("Models extracted:", known)
print("Unknown models:", unknown)
print(
    "Extraction coverage:",
    round(known / total * 100, 2) if total else 0,
    "%"
)

display(
    samsung_titles[
        ["title", "samsung_model"]
    ].head(20)
)

Review Unknown Samsung Models

In [ ]:
unknown_samsung = samsung_titles.loc[
    samsung_titles["samsung_model"] == "unknown",
    ["title"]
]

print("Unknown Samsung ads:", len(unknown_samsung))
display(unknown_samsung)

Save Samsung Models to Stage 1 Dataset

In [ ]:
samsung_mask = stage1_df["brand"] == "samsung"
stage1_df.loc[samsung_mask, "samsung_model"] = (
    stage1_df.loc[samsung_mask, "title"].apply(extract_samsung_model)
)
stage1_df["samsung_match_rule"] = ""
stage1_df.loc[samsung_mask & stage1_df["samsung_model"].ne("unknown"), "samsung_match_rule"] = "samsung:ordered_regex"
print("Samsung model column added successfully.")
print("Samsung rows:", int(samsung_mask.sum()))
print("Samsung unknown:", int((stage1_df.loc[samsung_mask, "samsung_model"] == "unknown").sum()))


Review Samsung Model Distribution

In [ ]:
display(
    stage1_df.loc[
        stage1_df["brand"] == "samsung",
        "samsung_model"
    ].value_counts().head(30)
)

Identify Samsung Parts and Accessories Listings

In [ ]:
parts_pattern = (
    r"شاشه|شاشة|قطع غيار|بورده|بوردة|بطاريه|بطارية|"
    r"جراب|اسكرينه|اسكرينة|شاحن|سماعه|سماعة|"
    r"screen|spare parts|battery|charger|case|cover"
)

samsung_parts = stage1_df.loc[
    (stage1_df["brand"] == "samsung")
    & stage1_df["title"].str.contains(
        parts_pattern, case=False, na=False, regex=True
    ),
    ["title", "samsung_model"]
]

print("Possible parts/accessories:", len(samsung_parts))
display(samsung_parts.head(30))

Mark Non-Phone Listings

In [ ]:
parts_only_pattern = (
    r"^\s*(?:شاشه|شاشة|بطاريه|بطارية|بورده|بوردة|"
    r"جراب|اسكرينه|اسكرينة|شاحن|سماعه|سماعة)\b"
    r"|محتاج شاشه|محتاج شاشة|مطلوب شاشه|مطلوب شاشة"
    r"|screen\s+(?:samsung|for\s+samsung)"
)

stage1_df["is_phone_listing"] = True

parts_mask = (
    (stage1_df["brand"] == "samsung")
    & stage1_df["title"].str.contains(
        parts_only_pattern, case=False, na=False, regex=True
    )
)

stage1_df.loc[parts_mask, "is_phone_listing"] = False

print("Non-phone listings marked:", (~stage1_df["is_phone_listing"]).sum())

display(
    stage1_df.loc[
        ~stage1_df["is_phone_listing"],
        ["title", "brand", "samsung_model"]
    ]
)

Review Brand Distribution in Training Data

In [ ]:
print("Brands in training data:")
display(stage1_df["brand"].value_counts(dropna=False))

# Xiaomi

Some helper functions

In [ ]:
def explore_brand(brand, n=20, seed=10):
    brand_titles = stage1_df.loc[stage1_df["brand"] == brand].copy()
    print(f"Total {brand} ads:", len(brand_titles))
    display(
        brand_titles[["title", "title_punct_normalized"]].sample(
            n=min(n, len(brand_titles)), random_state=seed
        )
    )


def evaluate_brand(brand, show_rows=20, max_unknown=20):
    col = f"{brand}_model"
    rule_col = f"{brand}_match_rule"
    mask = stage1_df["brand"] == brand
    details = stage1_df.loc[mask, "title"].apply(
        lambda title: extract_brand_model_with_rule(title, brand)
    )
    stage1_df.loc[mask, col] = [item[0] for item in details]
    stage1_df.loc[mask, rule_col] = [item[1] for item in details]

    results = stage1_df.loc[mask, col]
    total = len(results)
    known = int(results.ne("unknown").sum())
    coverage = known / total * 100 if total else 0
    print(f"Brand: {brand}")
    print(f"Total ads: {total}")
    print(f"Models extracted: {known}")
    print(f"Unknown models: {total - known}")
    print(f"Extraction coverage: {coverage:.2f}%")


In [ ]:
explore_brand("xiaomi")

# Define Xiaomi Model Rules

In [ ]:
ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
BRAND_WORDS = {
    "شاومي": "xiaomi", "ريدمي": "redmi", "ردمي": "redmi", "بوكو": "poco",
    "اوبو": "oppo", "رينو": "reno", "فايند": "find",
    "ريلمي": "realme", "ريالمي": "realme", "نارزو": "narzo",
    "هونر": "honor", "هونور": "honor", "اونر": "honor", "ماجيك": "magic",
    "انفينكس": "infinix", "انفنكس": "infinix", "فيفو": "vivo", "هواوي": "huawei",
    "نوفا": "nova", "موتورولا": "motorola", "موتو": "moto", "ريزر": "razr",
    "ون بلس": "oneplus", "وان بلس": "oneplus", "ونبلس": "oneplus", "نورد": "nord",
    "تكنو": "tecno", "سبارك": "spark", "كامون": "camon", "فانتوم": "phantom",
    "بوفا": "pova", "ايتل": "itel", "جوجل": "google", "بيكسل": "pixel",
    "سوني": "sony", "اكسبيريا": "xperia", "اكسبريا": "xperia", "نوكيا": "nokia",
    "لينوفو": "lenovo", "انفنيكس": "infinix", "نوبيا": "nubia", "بورا": "pura",
    "ستايلس": "stylus", "رازر": "razr", "بكسل": "pixel",
}
SERIES_WORDS = {
    "برو": "pro", "بلس": "plus", "الترا": "ultra", "ماكس": "max", "لايت": "lite",
    "نوت": "note", "ميني": "mini", "فولد": "fold", "فليب": "flip",
    "بلاي": "play", "برايم": "prime", "هوت": "hot", "سمارت": "smart", "ايدج": "edge",
}
ARABIC_RANGE = "\u0600-\u06FF"


def prepare_text(title, letters=None, aliases=None, pre=None):
    text = str(title).lower().translate(ARABIC_DIGITS)
    text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)
    for ch in "أإآٱ":
        text = text.replace(ch, "ا")
    text = text.replace("ى", "ي").replace("+", " plus ").replace("_", " ")
    text = re.sub(r"[^\w\s.]", " ", text)
    text = re.sub(r"(?<!\d)\.|\.(?!\d)", " ", text)
    text = re.sub(
        rf"(?<=[{ARABIC_RANGE}])(?=[a-z0-9])|(?<=[a-z0-9])(?=[{ARABIC_RANGE}])",
        " ", text
    )
    all_brand_words = {**BRAND_WORDS, **(aliases or {})}
    for old in sorted(all_brand_words, key=len, reverse=True):
        text = text.replace(old, f" {all_brand_words[old]} ")
    for old, new in SERIES_WORDS.items():
        text = re.sub(rf"(?<![{ARABIC_RANGE}]){old}(?![{ARABIC_RANGE}])", f" {new} ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\bnot\s*(?=\d)", "note ", text)
    text = re.sub(r"(?<=\d)(pro|max|plus|ultra|lite|mini)\b", r" \1", text)
    text = re.sub(r"(?<=\d)bro\b", " pro", text)
    text = re.sub(r"(?<=[a-z])5g\b", " 5g", text)
    for pattern, repl in (pre or []):
        text = re.sub(pattern, repl, text)
    for old, new in (letters or {}).items():
        text = re.sub(rf"(?<!\S){old}\s*(?=\d)", f" {new} ", text)
        text = re.sub(rf"(?<=\d)\s*{old}(?![{ARABIC_RANGE}])", f"{new} ", text)
    return re.sub(r"\s+", " ", text).strip()


BRAND_CONFIG = {}


def extract_brand_model_with_rule(title, brand):
    cfg = BRAND_CONFIG[brand]
    raw = str(title).lower()
    if len(brand) >= 4:
        raw = re.sub(rf"({brand})(?=[a-z0-9])", r"\1 ", raw)
    text = prepare_text(raw, cfg.get("letters"), cfg.get("aliases"), cfg.get("pre"))
    for rule_index, (pattern, template) in enumerate(cfg["rules"], start=1):
        match = re.search(pattern, text)
        if match:
            groups = [group or "" for group in match.groups()]
            model = template.format("", *groups)
            model = re.sub(r"\s+", " ", model).strip()
            return f"{brand} {model}".strip(), f"{brand}:rule_{rule_index:02d}"
    return "unknown", ""


def extract_brand_model(title, brand):
    return extract_brand_model_with_rule(title, brand)[0]


In [ ]:

BRAND_CONFIG["xiaomi"] = {
    "letters": {
        "اكس": "x",
        "اف": "f",
        "ام": "m",
        "سي": "c",
        "كيه": "k",
        "ايه": "a",
        "تي": "t",
        "مي": "mi",
    },

    "aliases": {
        "ميكس": "mix",
        "تربو": "turbo",
        "بلاك شارك": "black shark",
    },

    "pre": [
        (r"xiaomi\s+مي\s+تن\s+تي", "xiaomi mi 10t"),
        (r"xiaomi\s+بوك\s+اكس\s+ثري\s+pro", "poco x3 pro"),
        (r"xiaomi\s+بوك\s+اكس\s+ثري", "poco x3"),
        (r"\bxiaomi\s+18\s*fold\b", "xiaomi 18 fold"),
        (r"\bxiaomi\s+fold\s+xiaomi\s+18\s*fold\b",
         "xiaomi fold xiaomi 18 fold"),
    ],

    "rules": [
        # Black Shark
        (
            r"\bblack\s*shark\s*(\d{1,2})(?:\s*(pro))?\b",
            "black shark {1} {2}"
        ),

        (
            r"\bredmi\s*note\s*(\d{1,2})"
            r"\s*(s|t|a)?"
            r"\s*(pro\s*plus|pro\s*max|pro|plus)?"
            r"\s*(5g)?\b",
            "redmi note {1}{2} {3} {4}"
        ),

        # Redmi K
        (
            r"\bredmi\s*k\s*(\d{2,3})"
            r"\s*(pro\s*max|pro|ultra|plus)?\b",
            "redmi k{1} {2}"
        ),

        # Redmi Turbo
        (
            r"\bredmi\s*turbo\s*(\d{1,2})\s*(pro|max)?\b",
            "redmi turbo {1} {2}"
        ),

        # Redmi A / C
        (
            r"\bredmi\s*([ac])\s*(\d{1,2})(y)?\b",
            "redmi {1}{2}{3}"
        ),

        # Redmi 12C / 13C / 12 5G / 13 5G وما شابه
        (
            r"\bredmi\s*(\d{1,2})\s*(c|a|t|s|x)"
            r"\s*(prime|pro|plus)?\b",
            "redmi {1}{2} {3}"
        ),

        (
            r"\bredmi\s*(\d{1,2})"
            r"\s*(prime|pro|plus)?\b",
            "redmi {1} {2}"
        ),

        # POCO X / F / M / C
        (
            r"\bpoco\s*([xfmc])\s*(\d{1,2})"
            r"\s*(pro|max|gt|ultra|plus)?\b",
            "poco {1}{2} {3}"
        ),

        (
            r"\bpoco\b.*?\b([xfmc])\s*(\d{1,2})"
            r"\s*(pro|max|gt|ultra|plus)?\b",
            "poco {1}{2} {3}"
        ),

        # Xiaomi Mix Fold / Flip
        (
            r"\bmix\s*(fold|flip)\s*(\d{1,2})?\b",
            "mix {1} {2}"
        ),

        # Xiaomi 18 Fold أو Xiaomi 18fold
        (
            r"\bxiaomi\s*(\d{1,2})\s*fold\b",
            "{1} fold"
        ),

        # Mi A
        (
            r"\bmi\s*a\s*(\d{1,2})\b",
            "mi a{1}"
        ),

        # Mi Note
        (
            r"\bmi\s*note\s*(\d{1,2})\s*(pro|lite)?\b",
            "mi note {1} {2}"
        ),

        # Mi / Mi T
        (
            r"\bmi\s*(\d{1,2}t?)\s*(pro|lite|ultra)?\b",
            "mi {1} {2}"
        ),

        # Xiaomi 12C -> Redmi 12C
        (
            r"\bxiaomi\s*(\d{1,2})\s*c\b",
            "redmi {1}c"
        ),

        # Xiaomi A3 -> Redmi A3
        (
            r"\bxiaomi\s*a\s*(\d{1,2})\b",
            "redmi a{1}"
        ),

        (
            r"\bxiaomi\s*(\d{1,2}t?)"
            r"\s*(pro|lite|ultra|plus)?\b",
            "{1} {2}"
        ),

        # Ultra 15 -> 15 ultra
        (
            r"\bultra\s*(\d{2})\b",
            "{1} ultra"
        ),

        (
            r"\bc\s*(\d{2})\b",
            "redmi c{1}"
        ),

        (
            r"\bnote\s*(\d{1,2})"
            r"\s*(s|t)?"
            r"\s*(pro\s*plus|pro\s*max|pro|plus)?"
            r"\s*(5g)?\b",
            "redmi note {1}{2} {3} {4}"
        ),
    ],
}


In [ ]:
evaluate_brand("xiaomi")

In [ ]:
stage1_df.loc[
    stage1_df["xiaomi_model"].eq("unknown"),
    ["title", "title_normalized", "xiaomi_model"]
].head(69)

# Oppo

In [ ]:
explore_brand("oppo")

In [ ]:

BRAND_CONFIG["oppo"] = {
    "letters": {
        "ايه": "a",
        "اي": "a",
        "اية": "a",
        "اف": "f",
        "ار": "r",
        "تي": "t",
        "اكس": "x",
    },

    "aliases": {
        "opoo": "oppo",
        "oppooo": "oppo",
        "فايف جي": "5g",
        "فايند": "find",
    },

    "pre": [
        (r"\bfind\s+ان\s*(?=\d)", "find n "),

        # Turbo: تربو -> turbo
        (
            r"(?<![\u0600-\u06FF])تربو(?![\u0600-\u06FF])",
            "turbo"
        ),

        (r"\ba\s*6\s+اي\b", "a6i"),

        # A6 Pro
        (r"\ba\s*6\s+pro\b", "a6pro"),
    ],

    "rules": [
        (r"\ba\s*6a\b", "a6i"),

        # Find N series
        (
            r"\bfind\s*n\s*(\d{1,2})\b",
            "find n{1}"
        ),

        # Find X series
        (
            r"\bfind\s*x\s*(\d{1,2})\s*(pro|plus|ultra)?\b",
            "find x{1} {2}"
        ),

        # K series: K13 Turbo Pro 5G
        (
            r"\bk\s*(\d{1,2})\s*turbo\s*(pro)?\s*(5g)?\b",
            "k{1} turbo {2} {3}"
        ),

        # Reno series: Reno 7, Reno 8T, Reno 15F 5G
        (
            r"\breno\s*(\d{1,2})\s*(t|f|z)?\s*"
            r"(pro\s*plus|pro|plus)?\s*(5g)?\b",
            "reno {1}{2} {3} {4}"
        ),

        # A series: A60, A5i, A6 Pro, A6X, A1K
        (
            r"\ba\s*(\d{1,3})\s*"
            r"(pro\s*plus|pro|plus|x|k|s|i)?\s*(5g)?\b",
            "a{1}{2} {3}"
        ),

        # F series: F5, F1s
        (
            r"\bf\s*(\d{1,2})\s*(s|x|pro)?\b",
            "f{1}{2}"
        ),

        # R series: R15 Pro
        (
            r"\br\s*(\d{1,2})\s*(pro|plus)?\b",
            "r{1} {2}"
        ),
    ]
}





In [ ]:
evaluate_brand("oppo")

In [ ]:
stage1_df.loc[
    stage1_df["oppo_model"].eq("unknown"),
    ["title", "title_normalized", "oppo_model"]
].head(69)

# honor

In [ ]:
explore_brand("honor")

In [ ]:

BRAND_CONFIG["honor"] = {
    "letters": {
        "اكس": "x",
        "سي": "c",
        "سى": "c",
        "سيه": "c",
        "سيي": "c",
        "بي": "b",
        "بى": "b",
        "ايه": "a",
        "اي": "i",
    },

    "aliases": {
        "honour": "honor",
        "هونور": "honor",
        "اونر": "honor",
    },

    "pre": [
        # Arabic X: اكس 8 -> x 8
        (r"(?<![\u0600-\u06FF])اكس(?![\u0600-\u06FF])", "x"),

        # Arabic Magic V: ماجيك في 3 -> magic v 3
        (r"ماجيك\s+في(?=\s*\d)", "magic v"),

        # Correct likely typo only in the specific model context
        (r"\bhonor\s+wen\s+rt\b", "honor win rt"),
        (r"\bhonor\s+wen(?=\s+5g\b)", "honor win"),
    ],

    "rules": [
        # Honor X series: X6, X6b, X7a, X8c, X9c
        (r"\bx\s*(\d{1,2})\s*(c|b|a|i|d|pro|plus)?\b", "x{1}{2}"),

        # Arabic reversed format: 6 X -> X6
        (r"\b(\d{1,2})\s*x\s*(c|b|a)?\b", "x{1}{2}"),

        # Magic V series with a version number
        (r"\bmagic\s+v\s*(\d{1,2})\s*(pro|plus)?\b", "magic v{1} {2}"),

        # Magic V without a version number
        (r"\bmagic\s+v\b", "magic v"),

        # Magic numeric series: Magic 5 Pro, Magic 8 Pro
        (r"\bmagic\s*(\d{1,2})\s*(pro\s*plus|pro|plus|lite)?\b", "magic {1} {2}"),

        # Honor numeric series: Honor 200 Pro, Honor 600
        (r"\bhonor\s+(\d{2,3})\s*(pro\s*plus|pro|plus|lite)?\b", "{1} {2}"),

        # Honor Win RT / Win
        (r"\bwin\s+rt\b", "win rt"),
        (r"\bwin\b", "win"),

        # Honor Robot Phone / Robot
        (r"\brobot(?:\s+phone)?\b", "robot"),

        # Honor Play
        (r"\bplay\b", "play"),

        # Honor GT
        (r"\bgt\b", "gt"),

        # Honor 8 Pro / 8 Plus / 8 Lite
        (r"\bhonor\s+(\d{1,2})\s+(pro|plus|lite)\b", "{1} {2}"),

        # Honor 8X / 8C
        (r"\bhonor\s+(\d{1,2})\s*(x|c)\b", "{1}{2}"),
    ],
}


In [ ]:
evaluate_brand("honor")

In [ ]:
stage1_df.loc[
    stage1_df["honor_model"].eq("unknown"),
    ["title", "title_normalized", "honor_model"]
].head(69)

# Realme

In [ ]:
explore_brand("realme")

In [ ]:

BRAND_CONFIG["realme"] = {
    "letters": {
        "سي": "c",
        "سى": "c",
        "بي": "p",
        "بى": "p",
        "س": "c",
    },

    "aliases": {
        "ريلمى": "realme",
        "نارزو": "narzo",
        "نوت": "note",
    },

    "pre": [
        # Arabic model names: specific phrases must come first
        (r"جي\s*تي\s*نيو", "gt neo"),
        (r"جي\s*تي", "gt"),
        (r"نوت(?=\s*\d)", "note"),
        (r"في(?=\s*\d)", "v"),

        # Fix common spelling variations
        (r"\bnot(?=\s*\d)", "note"),
        (r"\bgt(?=\d)", "gt "),
        (r"\bc(?=\d)", "c "),
        (r"\bp(?=\d)", "p "),
        (r"\ba(?=\d)", "a "),
        (r"\bproه\b", "pro"),
        (r"بروه\b", "pro"),
    ],

    "rules": [
        # GT Neo series — place before general GT rules
        (r"\bgt\s*neo\s*(\d{1,2})\s*(t)?\b", "gt neo {1}{2}"),

        # GT Master Edition
        (r"\bgt\s*master\s*edition\b", "gt master edition"),

        # GT Pro series
        (r"\bgt\s*(\d{1,2})\s*pro\b", "gt {1} pro"),

        # GT T series: GT 6T -> gt 6t
        (r"\bgt\s*(\d{1,2})\s*t\b", "gt {1}t"),

        # Standard GT series
        (r"\bgt\s*(\d{1,2})\b", "gt {1}"),

        # Note series
        (r"\bnote\s*(\d{2,3})\s*(x)?\b", "note {1}{2}"),

        # C series: C21Y, C100i, C3
        (r"\bc\s*(\d{1,3})\s*([a-z])?\b", "c{1}{2}"),

        # P series
        (r"\bp\s*(\d{1,2})\s*(ultra|pro\s*plus|pro|plus)?\b", "p{1} {2}"),

        # Narzo series
        (r"\bnarzo\s*(\d{1,3})\s*(pro|plus|x)?\b", "narzo {1} {2}"),

        # V series
        (r"\bv\s*(\d{1,2})\s*(t|i)?\b", "v{1}{2}"),

        # Numbered series with suffixes: Realme 9i, 14t, etc.
        (r"\brealme\s+(\d{1,2})\s*(pro\s*plus|pro|plus)\b", "{1} {2}"),
        (r"\brealme\s+(\d{1,2})\s*(i|t|x)\b", "{1}{2}"),
        (r"\brealme\s+(\d{1,2})\b", "{1}"),

        # A series
        (r"\ba\s*(\d{1,2})\s*(pro|plus)?\b", "a{1} {2}"),
    ],
}


In [ ]:
evaluate_brand("realme")

In [ ]:
stage1_df.loc[
    stage1_df["realme_model"].eq("unknown"),
    ["title", "title_normalized", "realme_model"]
].head(69)

# infinix

In [ ]:
explore_brand("infinix")

In [ ]:

BRAND_CONFIG["infinix"] = {
    "letters": {},

    "aliases": {
        "انفنيكس": "infinix",
        "انفنكس": "infinix",
        "انفينكس": "infinix",
        "إنفنيكس": "infinix",
        "سمارت": "smart",
        "سمار": "smart",
        "هوت": "hot",
        "نوت": "note",
        "زيرو": "zero",
        "ايدج": "edge",
        "برو": "pro",
        "بلاي": "play",
    },

    "pre": [
        # Arabic series names
        (r"جي\s*تي", "gt"),
        (r"اس(?=\s*\d)", "s"),
        (r"سمارت\s*زيرو", "smart zero"),

        # Smart 6 اي -> Smart 6i
        (r"\bsmart\s*(\d{1,2})\s*اي\b", r"smart \1i"),

        # Join brand and model if written together
        (r"\binfinix(?=[a-z])", "infinix "),

        # Separate series from numbers
        (r"\b(gt|smart|note|hot|zero|s)\s*(?=\d)", r"\1 "),

        # Separate suffixes from numbers
        (r"(?<=\d)(pro|plus|play|ultra|lite)\b", r" \1"),
    ],

    "rules": [
        # GT series
        (r"\bgt\s*(\d{1,2})\s*pro\b", "gt {1} pro"),
        (r"\bgt\s*(\d{1,2})\s*plus\b", "gt {1} plus"),
        (r"\bgt\s*(\d{1,2})\b", "gt {1}"),

        # Smart Zero
        (r"\bsmart\s+zero\b", "smart zero"),

        # Smart series
        (r"\bsmart\s*(\d{1,2})\s*i\b", "smart {1}i"),
        (r"\bsmart\s*(\d{1,2})\b", "smart {1}"),

        # Zero series: Zero 8i, Zero 30, etc.
        (r"\bzero\s*(\d{1,2})\s*(i|x|pro)?\b", "zero {1}{2}"),

        # Note Edge
        (r"\bnote\s*edge\b", "note edge"),

        # Note series: Note 50s, Note 60
        (r"\bnote\s*(\d{1,3})\s*(s|pro|plus|x|i)?\b", "note {1}{2}"),

        # Hot series with letter suffix: Hot 60i
        (r"\bhot\s*(\d{1,3})\s*(i|s|x)\b", "hot {1}{2}"),

        # Hot series with word suffix: Hot 11 Play, Hot 60 Pro
        (
            r"\bhot\s*(\d{1,3})\s*(play|pro\s*plus|pro|plus|ultra|lite)?\b",
            "hot {1} {2}"
        ),

        # S series: S4, S5
        (r"\bs\s*(\d{1,2})\s*(pro|plus|lite)?\b", "s{1} {2}"),
    ],
}


In [ ]:
evaluate_brand("infinix")

In [ ]:
stage1_df.loc[
    stage1_df["infinix_model"].eq("unknown"),
    ["title", "title_normalized", "infinix_model"]
].head(69)

# Vivo

In [ ]:
explore_brand("vivo")

In [ ]:

BRAND_CONFIG["vivo"] = {
    "letters": {},

    "aliases": {
        "فيفو": "vivo",
        "اكس": "x",
        "إكس": "x",
        "فولد": "fold",
        "اف اي": "fe",
        "إف إي": "fe",
        "واي": "y",
        "واى": "y",
        "اس": "s",
        "إس": "s",
        "برو": "pro",
        "لايت": "lite",
        "الترا": "ultra",
        "نيو": "neo",
    },

    "pre": [
        # Arabic V-series notation: في 70 -> v 70
        (r"في(?=\s*\d)", "v"),

        # Arabic iQOO spelling variants
        (r"اي\s*كيو\s*او(?:و)?", "iqoo"),

        # Arabic model suffixes
        (r"اف\s*اي\b", "fe"),
        (r"اي(?=\s|$)", "e"),

        # Smart spacing for concatenated brand and model
        (r"\bvivo(?=[a-z])", "vivo "),

        # X Fold 3 Pro and similar
        (r"\bx\s*fold\s*(?=\d)", "x fold "),

        # Separate series from model number
        (r"\b(x|y|v|s)\s*(?=\d)", r"\1 "),

        # Separate suffixes when written directly after digits
        (r"(?<=\d)(pro|plus|ultra|lite|fe)\b", r" \1"),
    ],

    "rules": [
        # iQOO Neo series
        (
            r"\biqoo\s*neo\s*(\d{1,3})\s*(pro\s*plus|pro|plus|ultra)?\b",
            "iqoo neo {1} {2}"
        ),

        # X Fold series: X Fold 3 Pro
        (
            r"\bx\s*fold\s*(\d{1,2})\s*(pro|plus|ultra)?\b",
            "x fold {1} {2}"
        ),

        # X series with FE suffix: X300 FE
        (
            r"\bx\s*(\d{2,3})\s*(fe|ultra|pro|plus|lite|e)?\b",
            "x{1} {2}"
        ),

        # V series with suffix: V23e, V27e, V29e
        (
            r"\bv\s*(\d{1,3})\s*(e|s|d)\s*(pro|plus|lite|ultra)?\b",
            "v{1}{2} {3}"
        ),

        # V series with Pro / Lite / Ultra
        (
            r"\bv\s*(\d{1,3})\s*(pro|plus|lite|ultra)\b",
            "v{1} {2}"
        ),

        # V series without suffix
        (
            r"\bv\s*(\d{1,3})\b",
            "v{1}"
        ),

        # Titles like "Vivo 40 Lite" or "Vivo 50 Lite"
        (
            r"\bvivo\s+(\d{1,3})\s*lite\b",
            "v{1} lite"
        ),

        # S series with suffix
        (
            r"\bs\s*(\d{1,3})\s*(pro|plus|lite|ultra)?\b",
            "s{1} {2}"
        ),

        # Y series with suffixes: Y11D, Y31D, Y20s, Y17s
        (
            r"\by\s*(\d{1,3})\s*(s|d|e)\s*(pro|plus|lite|ultra)?\b",
            "y{1}{2} {3}"
        ),

        # Y series with Pro / Lite / Plus
        (
            r"\by\s*(\d{1,3})\s*(pro|plus|lite|ultra)\b",
            "y{1} {2}"
        ),

        # Standard Y series
        (
            r"\by\s*(\d{1,3})\b",
            "y{1}"
        ),
    ],
}


In [ ]:
evaluate_brand("vivo")

In [ ]:
stage1_df.loc[
    stage1_df["vivo_model"].eq("unknown"),
    ["title", "title_normalized", "vivo_model"]
].head(69)

In [ ]:
explore_brand("huawei")

In [ ]:
BRAND_CONFIG["huawei"] = {
    "letters": {},

    "aliases": {
        "هواوي": "huawei",
        "هواوى": "huawei",
        "هواويه": "huawei",

        "ميت": "mate",
        "نوفا": "nova",
        "بورا": "pura",

        "برايم": "prime",
        "برو": "pro",
        "لايت": "lite",
        "بلس": "plus",
        "الترا": "ultra",
        "التميت": "ultimate",

        "بي": "p",
        "بى": "p",
        "واي": "y",
        "واى": "y",
        "اكس": "x",
        "إكس": "x",
        "سي": "se",
    },

    "pre": [
        (r"ميت\s*(?=[a-z0-9])", "mate "),
        (r"نوفا\s*(?=\d)", "nova "),
        (r"بورا\s*(?=\d)", "pura "),

        (r"\bmat(?=\s*\d)", "mate "),
        (r"\bhuawei(?=[a-z])", "huawei "),

        (r"\b(y|p|nova|mate|pura)\s*(?=\d)", r"\1 "),
        (r"\bmate\s*x\s*t\s*(?=\d)", "mate xt "),
        (r"\bmate\s*x\s*(?=\d)", "mate x "),

        (r"\by\s*(\d{1,2})([a-z])\b", r"y \1\2"),

        (r"\by\s*(9)\s*(2019)\b", r"y \1 \2"),

        (r"(?<=\d)(pro|max|plus|ultra|lite|prime|ultimate)\b", r" \1"),
    ],

    "rules": [
        # -------------------------
        # Huawei Pura series
        # -------------------------
        (r"\bpura\s*(\d{2})\s*s\s*pro\s*max\b",
         "pura {1}s pro max"),

        (r"\bpura\s*(\d{2})\s*pro\s*max\b",
         "pura {1} pro max"),

        (r"\bpura\s*(\d{2})\s*(ultra|pro|plus|lite|max)\b",
         "pura {1} {2}"),

        (r"\bpura\s*(\d{2})\b",
         "pura {1}"),

        # -------------------------
        # Huawei Mate XT series
        # -------------------------
        (r"\bmate\s*xt\s*(\d{1,2})\s*(ultimate|pro|plus)?\b",
         "mate xt{1} {2}"),

        # -------------------------
        # Huawei Mate X series
        # -------------------------
        (r"\bmate\s*x\s*(\d{1,2})\s*(ultimate|pro|plus)?\b",
         "mate x{1} {2}"),

        # -------------------------
        # Huawei Mate series
        # -------------------------
        (r"\bmate\s*(\d{1,2})\s*(pro|plus|lite|x)?\b",
         "mate {1} {2}"),

        # -------------------------
        # Huawei Nova series
        # -------------------------
        (r"\bnova\s*(\d{1,2})\s*(se|i|pro|plus|lite|max)?\b",
         "nova {1}{2}"),

        # -------------------------
        # Huawei P series
        # -------------------------
        (r"\bp\s*(\d{1,2})\s*(pro|plus|lite|max)?\b",
         "p{1} {2}"),

        # -------------------------
        # Huawei Y series
        # -------------------------
        (r"\by\s*(7)\s*a\b",
         "y{1}a"),

        (r"\by\s*(6)\s*p\b",
         "y{1}p"),

        (r"\by\s*(\d{1,2})\s*(prime)\s*(plus)\b",
         "y{1} prime plus"),

        (r"\by\s*(\d{1,2})\s*(prime)\s*(2019)\b",
         "y{1} prime {3}"),

        (r"\by\s*(\d{1,2})\s*(2019|2020)\b",
         "y{1} {2}"),

        (r"\by\s*(\d{1,2})\s*(prime|pro|plus|lite|p|s)?\b",
         "y{1} {2}"),
    ],
}

In [ ]:
evaluate_brand("huawei")

In [ ]:
stage1_df.loc[
    stage1_df["huawei_model"].eq("unknown"),
    ["title", "title_normalized", "huawei_model"]
].head(69)

# motorola

In [ ]:
explore_brand("motorola")

In [ ]:
BRAND_CONFIG["motorola"] = {
    "letters": {},

    "aliases": {
        # Brand
        "موتورولا": "motorola",
        "موتورله": "motorola",

        # Moto G
        "موتو": "moto",
        "جي": "g",
        "جى": "g",

        # Edge
        "ايدج": "edge",
        "إيدج": "edge",

        # Razr
        "رازر": "razr",
        "ريزر": "razr",

        # Stylus
        "ستايلس": "stylus",
        "ستايلوس": "stylus",

        # Other model words
        "فيوجن": "fusion",
        "الترا": "ultra",
        "زيروو": "zero",
    },

    "pre": [
        # Brand normalization
        (r"\bmotorola(?=[a-z0-9])", "motorola "),

        # Moto G series
        (r"\bmoto\s*g\s*(?=\d)", "moto g "),

        # Edge series
        (r"\bedge\s*(?=\d)", "edge "),

        # Razr series
        (r"\brazr\s*(?=\d)", "razr "),

        # Stylus series
        (r"\bstylus\s*(?=\d)", "stylus "),

        # G04, G24, etc.
        (r"\bg\s*(?=\d)", "g "),

        # Separate suffixes attached to digits
        (r"(?<=\d)(ultra|fusion|stylus)\b", r" \1"),
    ],

    "rules": [
        # -------------------------
        # Motorola Edge
        # -------------------------

        # Reversed title: 50 Edge Fusion
        (r"\b(\d{2})\s*edge\s*fusion\b",
         "edge {1} fusion"),

        # Edge 50 Fusion, Edge 60 Fusion
        (r"\bedge\s*(\d{2})\s*fusion\b",
         "edge {1} fusion"),

        # Edge models with suffix
        (r"\bedge\s*(\d{2})\s*(ultra|pro|neo|fusion|plus)\b",
         "edge {1} {2}"),

        # Edge models without suffix
        (r"\bedge\s*(\d{2})\b",
         "edge {1}"),

        # Edge 2023+
        (r"\bedge\s*(2023)\s*\+",
         "edge {1} plus"),

        (r"\bedge\s*(\d{4})\b",
         "edge {1}"),

        # -------------------------
        # Motorola Razr
        # -------------------------
        (r"\brazr\s*(\d{2})\s*(ultra|plus)?\b",
         "razr {1} {2}"),

        # -------------------------
        # Motorola Stylus
        # -------------------------
        (r"\bstylus\s*5g\b",
         "stylus 5g"),

        (r"\bstylus\s*(\d{1,2})\s*(5g)?\b",
         "stylus {1} {2}"),

        # -------------------------
        # Motorola Moto G
        # -------------------------

        # Moto G5 Plus, Moto G85, Moto G56, etc.
        (r"\bmoto\s*g\s*(\d{1,3})\s*(plus|power|play|pro)?\b",
         "moto g{1} {2}"),

        # G04, G24, G85, etc. without Moto
        (r"\bg\s*(\d{2,3})\s*(plus|power|play|pro)?\b",
         "g{1} {2}"),

        # -------------------------
        # Motorola Signature
        # -------------------------
        (r"\bsignature\b",
         "signature"),
    ],
}

In [ ]:
evaluate_brand("motorola")

In [ ]:
stage1_df.loc[
    stage1_df["motorola_model"].eq("unknown"),
    ["title", "title_normalized", "motorola_model"]
].head(69)

# oneplus

In [ ]:
explore_brand("oneplus")

In [ ]:
BRAND_CONFIG["oneplus"] = {
    "letters": {},

    "aliases": {
        # Brand names in Arabic
        "وان بلس": "oneplus",
        "ون بلس": "oneplus",
        "وانبلس": "oneplus",
        "ونبلس": "oneplus",

        # Series names
        "ايس": "ace",
        "آيس": "ace",
        "نورد": "nord",
        "نورد سي": "nord ce",
        "ار": "r",
        "تي": "t",

        # Suffixes
        "برو": "pro",
        "بلس": "plus",
        "الترا": "ultra",
        "اوبن": "open",
        "ماكلارين": "mclaren",
    },

    "pre": [
        # Normalize brand spelling
        (r"\bone\s+plus\b", "oneplus"),
        (r"\boneplus(?=[a-z0-9])", "oneplus "),

        # Normalize series names
        (r"\bace\s*(?=\d)", "ace "),
        (r"\bnord\s+ce\s*(?=\d)", "nord ce "),
        (r"\bnord\s*(?=\d)", "nord "),

        # Separate suffixes attached to digits
        (r"(?<=\d)(pro|plus|ultra)\b", r" \1"),

        # Normalize numbered suffixes
        (r"\b(\d{1,2})\s*t\s*pro\s*mclaren\b",
         r"\1t pro mclaren"),

        (r"\b(\d{1,2})\s*t\s*pro\b",
         r"\1t pro"),

        (r"\b(\d{1,2})\s*r\b",
         r"\1r"),
    ],

    "rules": [
        # -------------------------
        # OnePlus Ace
        # -------------------------
        (r"\bace\s*(\d{1,2})\s*t\b",
         "ace {1}t"),

        (r"\bace\s*(\d{1,2})\s*(pro|plus|ultra)\b",
         "ace {1} {2}"),

        (r"\bace\s*(\d{1,2})\b",
         "ace {1}"),

        # -------------------------
        # OnePlus Nord CE
        # -------------------------
        (r"\bnord\s*ce\s*(\d{1,2})\s*(lite)?\b",
         "nord ce{1} {2}"),

        # -------------------------
        # OnePlus Nord
        # -------------------------
        (r"\bnord\s*(\d{1,2})\s*(pro|plus)?\b",
         "nord {1} {2}"),

        # -------------------------
        # OnePlus Open
        # -------------------------
        (r"\bopen(?:\s+fold)?\b",
         "open"),

        # -------------------------
        # OnePlus numbered models
        # -------------------------

        # 7T Pro McLaren Edition
        (r"\b(\d{1,2})\s*t\s*pro\s*mclaren\b",
         "{1}t pro mclaren"),

        # 13s, 13t, 12r
        (r"\b(\d{1,2})\s*(s|t|r)\b",
         "{1}{2}"),

        # Numbered Pro models
        (r"\b(\d{1,2})\s*pro\b",
         "{1} pro"),

        # Numbered Plus / Ultra models
        (r"\b(\d{1,2})\s*(plus|ultra)\b",
         "{1} {2}"),

        # Numbered models without suffix
        (r"\b(\d{1,2})\b",
         "{1}"),
    ],
}

In [ ]:
evaluate_brand("oneplus")

# zte

In [ ]:
explore_brand("zte")

In [ ]:
BRAND_CONFIG["zte"] = {
    "letters": {},

    "aliases": {
        # Brand
        "زد تي اي": "zte",
        "زد تى اي": "zte",
        "زد تي إي": "zte",
        "زد تى إي": "zte",
        "زدتي اي": "zte",
        "زدتي": "zte",

        # Nubia
        "نوبيا": "nubia",
        "نوبيا زد": "nubia z",

        # RedMagic
        "ريد ماجيك": "red magic",
        "ريدماجيك": "red magic",

        # Neo
        "نيو": "neo",

        # Other
        "برو": "pro",
        "اير": "air",
        "إير": "air",
        "الترا": "ultra",
        "بلس": "plus",
    },

    "pre": [
        # Brand normalization
        (r"\bzte(?=[a-z0-9])", "zte "),

        # Normalize Nubia and RedMagic
        (r"\bred\s*magic\s*(?=\d)", "red magic "),
        (r"\bnubia\s*z\s*(?=\d)", "nubia z "),
        (r"\bnubia\s*a\s*(?=\d)", "nubia a "),
        (r"\bnubia\s*v\s*(?=\d)", "nubia v "),
        (r"\bnubia\s*neo\s*(?=\d)", "nubia neo "),

        # Blade series
        (r"\bblade\s*a\s*(?=\d)", "blade a "),

        # Normalize Neo series
        (r"\bneo\s*(?=\d)", "neo "),

        # Fix attached model names
        (r"\bnubia(?=[a-z]\d)", "nubia "),
    ],

    "rules": [
        # -------------------------
        # RedMagic
        # -------------------------
        (r"\bred\s*magic\s*(\d{1,2})\s*(pro|max|ultra|air)?\b",
         "red magic {1} {2}"),

        # -------------------------
        # Nubia Neo
        # -------------------------
        (r"\bnubia\s*neo\s*(\d{1,2})\s*(5g|pro|plus)?\b",
         "nubia neo {1} {2}"),

        # -------------------------
        # Nubia Z series
        # -------------------------
        (r"\bnubia\s*z\s*(\d{1,2})\s*(pro|ultra|plus)?\b",
         "nubia z{1} {2}"),

        # -------------------------
        # Nubia A series
        # -------------------------
        (r"\bnubia\s*a\s*(\d{1,3})\s*(5g)?\b",
         "nubia a{1} {2}"),

        # -------------------------
        # Nubia V series
        # -------------------------
        (r"\bnubia\s*v\s*(\d{1,3})\s*(5g)?\b",
         "nubia v{1} {2}"),

        # -------------------------
        # ZTE Blade
        # -------------------------
        (r"\bblade\s*a\s*(\d{1,3})\s*(5g)?\b",
         "blade a{1} {2}"),

        # -------------------------
        # ZTE Neo
        # -------------------------
        (r"\bneo\s*(\d{1,2})\b",
         "nubia neo {1}"),

        # -------------------------
        # Unknown / Other Model
        # -------------------------
        (r"\bother\s+model\b",
         "unknown"),
    ],
}

In [ ]:
evaluate_brand("zte")

In [ ]:
stage1_df.loc[
    stage1_df["zte_model"].eq("unknown"),
    ["title", "title_normalized", "zte_model"]
].head(69)

# tecno

In [ ]:
explore_brand("tecno")

In [ ]:
BRAND_CONFIG["tecno"] = {
    "letters": {},

    "aliases": {
        "تكنو": "tecno",

        "سبارك": "spark",
        "سبارك جو": "spark go",
        "سبارك سي ام": "spark cm",
        "سبارك سي إم": "spark cm",

        "كامون": "camon",
        "كامون سي ام": "camon cm",
        "كامون سي إم": "camon cm",

        "بوفا": "pova",
        "بوب": "pop",

        "برو": "pro",
        "بلس": "plus",
        "جو": "go",
    },

    "pre": [
        (r"\btecno(?=[a-z0-9])", "tecno "),

        (r"\bspark\s*go\s*(?=\d)", "spark go "),
        (r"\bspark\s*cm\b", "spark cm"),
        (r"\bspark\s*(?=\d)", "spark "),
        (r"\bcamon\s*cm\b", "camon cm"),
        (r"\bcamon\s*(?=\d)", "camon "),
        (r"\bpova\s*(?=\d)", "pova "),
        (r"\bpop\s*(?=\d)", "pop "),

        (r"(?<=\d)(pro|plus)\b", r" \1"),
    ],

    "rules": [
        # Tecno Spark Go 3
        (
            r"\bspark\s*go\s*(\d{1,2})\b",
            "spark go {1}"
        ),

        # Tecno Spark 20 Pro+
        (
            r"\bspark\s*(\d{1,3})\s*pro\s*plus\b",
            "spark {1} pro plus"
        ),

        # Tecno Spark 20 Pro
        (
            r"\bspark\s*(\d{1,3})\s*pro\b",
            "spark {1} pro"
        ),

        # Tecno Spark CM
        (
            r"\bspark\s*cm\b",
            "spark cm"
        ),

        # Tecno Spark 30C
        (
            r"\bspark\s*(\d{1,3})\s*c\b",
            "spark {1}c"
        ),

        # Tecno Spark 30 / Spark 40
        (
            r"\bspark\s*(\d{1,3})\b",
            "spark {1}"
        ),

        # Tecno Camon CM
        (
            r"\bcamon\s*cm\b",
            "camon cm"
        ),

        # Tecno Camon 40 Pro 5G
        (
            r"\bcamon\s*(\d{1,3})\s*pro\s*5g\b",
            "camon {1} pro 5g"
        ),

        # Tecno Camon 40 Pro
        (
            r"\bcamon\s*(\d{1,3})\s*pro\b",
            "camon {1} pro"
        ),

        # Tecno Camon 40
        (
            r"\bcamon\s*(\d{1,3})\b",
            "camon {1}"
        ),

        # Tecno Pova 6 Pro / Pova 7
        (
            r"\bpova\s*(\d{1,3})\s*(pro|plus|ultra)?\b",
            "pova {1} {2}"
        ),

        # Tecno Pop 7
        (
            r"\bpop\s*(\d{1,3})\b",
            "pop {1}"
        ),
    ],
}

In [ ]:
evaluate_brand("tecno")

# itel

In [ ]:
explore_brand("itel")

In [ ]:

BRAND_CONFIG["itel"] = {
    "letters": {},
    "aliases": {
        "ايتل": "itel",
        "ايتيل": "itel",

        "اس": "s",
        "إس": "s",

        "ايه": "a",
        "إيه": "a",

        "ار": "r",
        "آر": "r",

        "اس 4": "s4",
        "اس 24": "s24",
        "اس 25": "s25",

        "الترا": "ultra",
        "سي": "c",
        "سى": "c",
    },

    "pre": [
        (r"\bitel(?=[a-z0-9])", "itel "),
        (r"\b([as])\s*(\d{1,3})\s*c\b", r"\1 \2c"),
        (r"\ba\s*(?=\d)", "a "),
        (r"\bs\s*(?=\d)", "s "),
        (r"\brs\s*(?=\d)", "rs "),
        (r"\br\s+s\s*(?=\d)", "rs "),
        (r"\b(\d{1,3})\s*c\b", r"\1c"),
    ],

    "rules": [
        # itel S25 Ultra
        (
            r"\bs\s*(\d{1,3})\s*ultra\b",
            "s{1} ultra"
        ),

        # itel S24 / S25
        (
            r"\bs\s*(\d{1,3})\b",
            "s{1}"
        ),

        # itel A50C
        (
            r"\ba\s*(\d{1,3})\s*c\b",
            "a{1}c"
        ),

        # itel A90 / A80 / A100C
        (
            r"\ba\s*(\d{1,3})\s*c\b",
            "a{1}c"
        ),
        (
            r"\ba\s*(\d{1,3})\b",
            "a{1}"
        ),

        # itel RS4
        (
            r"\brs\s*(\d{1,3})\b",
            "rs {1}"
        ),
    ],
}


In [ ]:
evaluate_brand("itel")

In [ ]:
explore_brand("google")

In [ ]:

BRAND_CONFIG["google"] = {
    "letters": {},
    "aliases": {
        "جوجل": "google",
        "بكسل": "pixel",
        "برو": "pro",
        "فولد": "fold",
        "ايه": "a",
        "إيه": "a",
    },

    "pre": [
        (r"\bgoogle(?=[a-z0-9])", "google "),
        (r"\bpixel\s*(?=\d)", "pixel "),
        (r"\b(\d{1,2})\s*pro\s*fold\b", r"\1 pro fold"),
        (r"\b(\d{1,2})\s*pro\b", r"\1 pro"),
        (r"\b(\d{1,2})\s*5g\b", r"\1 5g"),
    ],

    "rules": [
        # Pixel 11 Pro Fold
        (
            r"\bpixel\s*(\d{1,2})\s*pro\s*fold\b",
            "{1} pro fold"
        ),

        # Pixel 6 Pro / Pixel 9 Pro
        (
            r"\bpixel\s*(\d{1,2})\s*pro\b",
            "{1} pro"
        ),

        # Pixel 8a / Pixel 9a
        (
            r"\bpixel\s*(\d{1,2})\s*a\b",
            "{1}a"
        ),

        # Pixel 10 5G
        (
            r"\bpixel\s*(\d{1,2})\s*5g\b",
            "{1} 5g"
        ),

        # Pixel 6 / Pixel 7 / Pixel 10 / Pixel 11
        (
            r"\bpixel\s*(\d{1,2})\b",
            "{1}"
        ),
    ],
}


In [ ]:
evaluate_brand("google")

# sony
11
nokia
9
lenovo
8
lg

In [ ]:
explore_brand("sony")

In [ ]:

BRAND_CONFIG["sony"] = {
    "letters": {},
    "aliases": {
        "سوني": "sony",
        "اكسبريا": "xperia",
        "إكسبريا": "xperia",
        "اريكسون": "ericsson",
        "إريكسون": "ericsson",
        "مارك": "mark",
        "برو": "pro",
        "الترا": "ultra",
        "زد": "z",
    },

    "pre": [
        (r"\bsony(?=[a-z0-9])", "sony "),
        (r"\bxperia(?=[a-z0-9])", "xperia "),
        (r"\b(\d)\s*mark\s*(\d)\b", r"\1 mark \2"),
        (r"\b(\d{1,2})\s*(iv|v|vi|iii|ii)\b", r"\1 \2"),
        (r"\bpro\s*[- ]?\s*i\b", "pro i"),
        (r"\bz\s*ultra\b", "z ultra"),
    ],

    "rules": [
        # Sony Xperia 1 Pro-I
        (
            r"\bxperia\s+pro\s+i\b",
            "xperia pro-i"
        ),

        # Sony Xperia 1 Mark 2 / Mark 7
        (
            r"\bxperia\s+(\d)\s*mark\s*(\d)\b",
            "xperia {1} mark {2}"
        ),

        # Sony Xperia 1 V / 5 IV
        (
            r"\bxperia\s+(\d{1,2})\s*(iv|v|vi|iii|ii)\b",
            "xperia {1} {2}"
        ),

        # Sony Xperia Z Ultra
        (
            r"\bxperia\s+z\s+ultra\b",
            "xperia z ultra"
        ),

        # Sony Xperia 1 / 5
        (
            r"\bxperia\s+(\d{1,2})\b",
            "xperia {1}"
        ),

        # Sony Ericsson K610i
        (
            r"\b([a-z]{1,3}\d{2,4}i?)\b",
            "{1}"
        ),
    ],
}


In [ ]:
evaluate_brand("sony")

# nokia

In [ ]:
explore_brand("nokia")

In [ ]:

BRAND_CONFIG["nokia"] = {
    "letters": {},
    "aliases": {
        "نوكيا": "nokia",
        "اكس": "x",
        "إكس": "x",
        "بلس": "plus",
        "برو": "pro",
        "الترا": "ultra",
    },

    "pre": [
        (r"\bnokia(?=[a-z0-9])", "nokia "),

        (r"\b([gc])\s*(?=\d)", r"\1 "),
        (r"\bx\s*(?=\d)", "x "),
        (r"\bn\s*(?=\d)", "n "),

        (r"\b(\d+)\s*\.\s*(\d+)\b", r"\1.\2"),

        (r"(?<=\d)(plus|pro|ultra)\b", r" \1"),
    ],

    "rules": [
        # Nokia G300 5G
        (
            r"\bg\s*(\d{1,3})\s*5g\b",
            "g{1} 5g"
        ),
        (
            r"\bg\s*(\d{1,3})\b",
            "g{1}"
        ),

        # Nokia C21 Plus / C-series
        (
            r"\bc\s*(\d{1,3})\s*plus\b",
            "c{1} plus"
        ),
        (
            r"\bc\s*(\d{1,3})\s*pro\b",
            "c{1} pro"
        ),
        (
            r"\bc\s*(\d{1,3})\b",
            "c{1}"
        ),

        # Nokia X350 / X2
        (
            r"\bx\s*(\d{1,4})\b",
            "x{1}"
        ),

        # Nokia N9
        (
            r"\bn\s*(\d{1,3})\b",
            "n{1}"
        ),

        # Nokia 7.1
        (
            r"\b(\d{1,2})\.(\d)\b",
            "{1}.{2}"
        ),
    ],
}




In [ ]:
evaluate_brand("nokia")

# lenovo

In [ ]:
explore_brand("lenovo")

In [ ]:

BRAND_CONFIG["lenovo"] = {
    "letters": {},
    "aliases": {
        "لينوفو": "lenovo",
        "كيه": "k",
        "نوت": "note",
        "نوتة": "note",
        "ليجن": "legion",
        "ليجيون": "legion",
        "بلس": "plus",
        "برو": "pro",
    },

    "pre": [
        (r"\blenovo(?=[a-z0-9])", "lenovo "),
        (r"\blegion\s*y\s*(?=\d)", "legion y "),
        (r"\bk\s*(?=\d)", "k "),
        (r"\ba\s*(?=\d)", "a "),
        (r"\b(note)\s*(?=\d)", "note "),
        (r"\b(\d{1,3})\s*note\b", r"\1 note"),
    ],

    "rules": [
        # Lenovo Legion Y70
        (
            r"\blegion\s*y\s*(\d{1,3})\b",
            "legion y{1}"
        ),

        # Lenovo K14 Plus
        (
            r"\bk\s*(\d{1,3})\s*plus\b",
            "k{1} plus"
        ),

        # Lenovo K12 Note / K8 Note
        (
            r"\bk\s*(\d{1,3})\s*note\b",
            "k{1} note"
        ),

        # Lenovo K15
        (
            r"\bk\s*(\d{1,3})\b",
            "k{1}"
        ),

        # Lenovo A8
        (
            r"\ba\s*(\d{1,3})\b",
            "a{1}"
        ),

        # Lenovo Note 6
        (
            r"\bnote\s*(\d{1,3})\b",
            "note {1}"
        ),
    ],
}


In [ ]:
evaluate_brand("lenovo")

In [ ]:
explore_brand("lg")

In [ ]:
BRAND_CONFIG["lg"] = {
    "letters": {},
    "aliases": {
        "ال جي": "lg",
        "إل جي": "lg",
        "ال جى": "lg",
        "إل جى": "lg",

        "فيلفيت": "velvet",
        "وينج": "wing",
        "وينغ": "wing",
        "ستايلو": "stylo",
        "ثينك كيو": "thinq",
        "ثينكيو": "thinq",
    },

    "pre": [
        (r"\blg(? = [a-z0-9])", "lg "),

        (r"\b(g|v|k|q|w|x)\s*(? = \d)", r"\1 "),
        (r"\bstylo\s*(? = \d)", "stylo "),
        (r"\bvelvet(? = \d)", "velvet "),
        (r"\bwing(? = \d)", "wing "),

        (r"\bthin\s*q\b", "thinq"),
        (r"\bthinq\b", "thinq"),

        (r"\b(v\d{1, 2})\s*thinq\b", r"\1 thinq"),
    ],

    "rules": [

        (r"\bvelvet\s*(\d{1, 2})?\b", "velvet {1}"),

        (r"\bwing\s*(\d{1, 2})?\b", "wing {1}"),

        (r"\bstylo\s*(\d{1, 2})\b", "stylo {1}"),

        (r"\bv\s*(\d{1, 2})\s*thinq\b", "v{1} thinq"),

        (r"\bg\s*(\d{1, 2})\b", "g{1}"),

        (r"\bv\s*(\d{1, 2})\b", "v{1}"),

        (r"\bk\s*(\d{1, 3})\b", "k{1}"),

        (r"\bq\s*(\d{1, 3})\b", "q{1}"),

        (r"\bw\s*(\d{1, 3})\b", "w{1}"),

        (r"\bx\s*(\d{1, 3})\b", "x{1}"),
    ],
}


In [ ]:
evaluate_brand("lg")

In [ ]:
print("Columns names:\n", stage1_df.columns.to_list())

### Repository integration additions — added during cleanup

The following cells combine per-brand predictions into a shared `model_key`, create the hand-label template, and calculate metrics from human gold labels. They were added after Fadwa’s original rule notebook.


## Unified model key, gold-set template and evaluation

`confidence` remains blank until it is calibrated against human gold labels. The template is sampled by brand and never invents labels.


In [1]:
brand_model_columns = {
    "apple": ("iphone_model", "iphone_match_rule"),
    "samsung": ("samsung_model", "samsung_match_rule"),
    **{brand: (f"{brand}_model", f"{brand}_match_rule") for brand in [
        "xiaomi", "oppo", "honor", "realme", "infinix", "vivo", "huawei",
        "motorola", "oneplus", "zte", "tecno", "itel", "google", "sony", "nokia", "lenovo", "lg"
    ]},
}
stage1_df["model_key"] = "unknown"
stage1_df["match_rule"] = ""
stage1_df["confidence"] = pd.Series(pd.NA, index=stage1_df.index, dtype="Float64")
for brand, (model_col, rule_col) in brand_model_columns.items():
    if model_col not in stage1_df:
        continue
    mask = stage1_df["brand"].eq(brand)
    values = stage1_df.loc[mask, model_col].fillna("unknown").astype(str).str.strip()
    known = values.str.casefold().ne("unknown") & values.str.casefold().ne("not_apple")
    if brand == "apple":
        stage1_df.loc[mask & known, "model_key"] = "apple " + values.loc[known]
    else:
        stage1_df.loc[mask & known, "model_key"] = values.loc[known]
    if rule_col in stage1_df:
        stage1_df.loc[mask, "match_rule"] = stage1_df.loc[mask, rule_col].fillna("")

stage_output_path = stage1_output_dir / "stage1_model_keys.csv"
coverage = (
    stage1_df.assign(_brand=stage1_df["brand"].fillna("unknown"), _matched=stage1_df["model_key"].ne("unknown"))
    .groupby("_brand", dropna=False)["_matched"]
    .agg(total="size", matched="sum")
    .reset_index()
    .rename(columns={"_brand": "brand"})
)
coverage["coverage"] = coverage["matched"] / coverage["total"]
stage1_df.to_csv(stage_output_path, index=False)
coverage.to_csv(stage1_output_dir / "coverage_by_brand.csv", index=False)
print("Saved S1 predictions:", stage_output_path)
display(coverage)

# Create a reproducible 200-title stratified labeling template once.
labels_dir = PROJECT_ROOT / "data" / "labels"
labels_dir.mkdir(parents=True, exist_ok=True)
gold_path = labels_dir / "gold_set_template.csv"
pred_columns = ["ad_id", "title", "brand", "model_key", "match_rule", "confidence"]
pred_for_gold = stage1_df[pred_columns].rename(columns={
    "model_key": "predicted_model_key", "match_rule": "predicted_match_rule"
})
if not gold_path.exists():
    sample_size = min(200, len(pred_for_gold))
    strata = stage1_df["brand"].fillna("unknown").astype(str)
    counts = strata.value_counts().to_dict()
    raw_alloc = {key: sample_size * count / len(strata) for key, count in counts.items()}
    allocation = {key: min(counts[key], max(1, int(raw_alloc[key]))) for key in counts}
    while sum(allocation.values()) > sample_size:
        candidates = [key for key in allocation if allocation[key] > 1]
        key = min(candidates, key=lambda item: raw_alloc[item] - allocation[item])
        allocation[key] -= 1
    while sum(allocation.values()) < sample_size:
        candidates = [key for key in allocation if allocation[key] < counts[key]]
        key = max(candidates, key=lambda item: raw_alloc[item] - allocation[item])
        allocation[key] += 1
    sample_parts = []
    strata_frame = pred_for_gold.assign(_stratum=strata)
    for key, count in allocation.items():
        sample_parts.append(strata_frame.loc[strata_frame["_stratum"].eq(key)].sample(n=count, random_state=42))
    gold = pd.concat(sample_parts, ignore_index=True).drop(columns="_stratum")
    gold["gold_model_key"] = ""
    gold["review_notes"] = ""
    gold.to_csv(gold_path, index=False)
else:
    gold = pd.read_csv(gold_path, keep_default_na=False)
    if "gold_model_key" not in gold:
        gold["gold_model_key"] = ""
    if "review_notes" not in gold:
        gold["review_notes"] = ""
    # Refresh predictions while preserving all human labels and notes.
    gold = gold.drop(columns=[c for c in ["predicted_model_key", "predicted_match_rule", "confidence"] if c in gold])
    gold = gold.merge(pred_for_gold.drop(columns=["title", "brand"]), on="ad_id", how="left")
    gold.to_csv(gold_path, index=False)
print("Gold-label template:", gold_path, "| rows:", len(gold))

labeled = gold.loc[gold["gold_model_key"].astype(str).str.strip().ne("")].copy()
if not labeled.empty:
    labeled["_gold"] = labeled["gold_model_key"].map(normalize_title_for_group)
    labeled["_pred"] = labeled["predicted_model_key"].fillna("unknown").map(normalize_title_for_group)
    covered = labeled["_pred"].ne("unknown")
    labeled["_correct"] = labeled["_pred"].eq(labeled["_gold"])
    overall = pd.DataFrame([{
        "labeled_rows": len(labeled),
        "coverage": float(covered.mean()),
        "precision_on_covered": float(labeled.loc[covered, "_correct"].mean()) if covered.any() else np.nan,
        "correct_rows": int(labeled["_correct"].sum()),
        "covered_rows": int(covered.sum()),
    }])
    overall.to_csv(stage1_output_dir / "gold_set_metrics.csv", index=False)
    labeled["_brand"] = labeled["brand"].fillna("unknown")
    labeled["_covered"] = covered
    by_brand = labeled.groupby("_brand").agg(
        labeled_rows=("_correct", "size"),
        covered_rows=("_covered", "sum"),
        correct_rows=("_correct", "sum"),
    ).reset_index().rename(columns={"_brand": "brand"})
    by_brand["coverage"] = by_brand["covered_rows"] / by_brand["labeled_rows"]
    by_brand["precision_on_covered"] = by_brand["correct_rows"] / by_brand["covered_rows"].replace(0, np.nan)
    by_brand.to_csv(stage1_output_dir / "gold_set_metrics_by_brand.csv", index=False)
    labeled.loc[~labeled["_correct"]].to_csv(stage1_output_dir / "gold_set_errors.csv", index=False)
    rule_scores = (
        labeled.loc[covered].groupby("predicted_match_rule")["_correct"]
        .agg(support="size", precision="mean").reset_index()
    )
    rule_scores.to_csv(stage1_output_dir / "rule_confidence.csv", index=False)
    calibrated = rule_scores.loc[rule_scores["support"] >= 5].set_index("predicted_match_rule")["precision"]
    stage1_df["confidence"] = stage1_df["match_rule"].map(calibrated).astype("Float64")
    stage1_df.to_csv(stage_output_path, index=False)
    print("Gold-set evaluation (confidence calibrated only for rules with >=5 labels):")
    display(overall)
else:
    print("Gold labels are blank. Fill gold_model_key by hand, then rerun this notebook to calculate precision and rule confidence.")

review = stage1_df.loc[
    stage1_df["model_key"].eq("unknown")
    | (stage1_df["confidence"].notna() & stage1_df["confidence"].lt(0.90))
].copy()
review.assign(_priority=review["confidence"].fillna(-1)).sort_values("_priority").drop(columns="_priority").head(100).to_csv(stage1_output_dir / "unknown_low_confidence_review.csv", index=False)
print("Review queue saved:", stage1_output_dir / "unknown_low_confidence_review.csv")


NameError: name 'stage1_df' is not defined